In [0]:
%run ../Includes/common_functions

In [0]:
%run ../Includes/config

In [0]:
v_data_source = dbutils.widgets.get("p_data_source")
v_file_date = dbutils.widgets.get("p_file_date")
print(v_data_source)
print(v_file_date)
print(raw_folder_path)

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType
pit_stops_schema = StructType(fields=[StructField("raceId", StringType(), False),
                                      StructField("driverId", StringType(), True),
                                      StructField("stop", StringType(), True),
                                      StructField("lap", StringType(), True),
                                      StructField("time", StringType(), True),
                                      StructField("duration", StringType(), True),
                                      StructField("milliseconds", StringType(), True)
                                     ])

pit_stops_df = (spark.read 
                    .schema(pit_stops_schema) 
                    .format("json")
                    .option("multiLine", True) 
                    .load(f"{raw_folder_path}/{v_file_date}/pit_stops.json"))

In [0]:
from pyspark.sql.functions import current_timestamp,lit

final_df = (pit_stops_df.withColumnRenamed("driverId", "driver_Id") 
                        .withColumnRenamed("raceId", "race_id") \
                        .withColumn("ingestion_date", current_timestamp())
                        .withColumn("data_source", lit(v_data_source)) 
                        .withColumn("file_date", lit(v_file_date))
)
                        
(final_df.write.mode("overwrite")
                .format("delta")
                .option("mergeSchema", True)
                .saveAsTable("f1.Bronze.pit_stops"))